# Preprocessing – SINTEF Unit Commitment (Tokke–Vinje)

**Group:** `[XX] <Kaggle team name>`
**Members:** `<Full name> (<student id>)`, `<Full name> (<student id>)`, `<Full name> (<student id>)`

This notebook turns the raw files in `data/` into model-ready tables. One **case** = one weekly
scheduling problem (168 hours) starting at `starttime` (UTC 00:00). For each case we must predict the
ON/OFF probability of 14 generating units for every hour → 14 × 168 = 2352 outputs per case.

* Train: Run No 1–2922 (2015-01-01 … 2022-12-31) – labels in `Unit_commitment_decisions.csv`
* Test: Run No 2923–3647 (2023-01-01 … 2024-12-25) – `prediction_mapping.csv`
* Metric: micro-averaged ROC-AUC over all 2352 columns (`kaggle_metric.py`)

**Outputs** (written to `processed/` in the project root; this notebook lives in `notebooks/`):

| file | granularity | content |
|---|---|---|
| `cases.csv` | 1 row per case (train + test) | static/case-level features (initial state, water values, weekly aggregates), `split`, `fold` |
| `hourly_train.csv` | 1 row per (case, hour), train | hourly features + 14 target columns `y_<unit>` |
| `hourly_test.csv` | 1 row per (case, hour), test | hourly features (no targets) |

The hourly files do **not** repeat the case features (that would make them ~10× larger) – join on `Run No`:

```python
cases = pd.read_csv("../processed/cases.csv")
train = pd.read_csv("../processed/hourly_train.csv").merge(cases, on="Run No")
```

For sequence models: rows are sorted by (`Run No`, `t`), so `train[cols].to_numpy().reshape(-1, 168, len(cols))`
gives a `(cases, 168, features)` tensor.
| `meta.json` | – | unit order, target column order for the submission, feature lists |

> Note: the short submission notebooks must start from raw data, so the functions here are written to be
> copy-pasted into them.

In [ ]:
import json, re
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

ROOT = Path("..")            # project root (this notebook is in notebooks/)
DATA = ROOT / "data"
KERNEL, EXTENDED = DATA / "kernel", DATA / "extended"
OUT = ROOT / "processed"
OUT.mkdir(exist_ok=True)

H = 168                      # hours per case
TZ_LOCAL = "Europe/Oslo"     # market behaviour follows local time
# The volume file contains the *actual* storage for every day, so the end-of-week volume of a case is
# known for both train and test. It is strong information but would not be known in real operations.
# Toggle to compare models with / without it.
USE_END_OF_WEEK_VOLUME = True

## 1. Load raw data

In [ ]:
def read_ts(path):
    df = pd.read_csv(path)
    df["date"] = pd.to_datetime(df["date"], utc=True)
    return df.set_index("date").sort_index()

price_raw  = read_ts(KERNEL / "Historical_day_ahead_price_2015_2025.csv")
inflow_raw = read_ts(KERNEL / "Historical_inflow_1958_2025.csv")
volume_raw = read_ts(KERNEL / "Historical_volume_2015_2024.csv")
wv_raw     = read_ts(KERNEL / "Synthetic_water_value_2015_2024.csv")
minvol_raw = read_ts(EXTENDED / "Constraint_min_volume.csv")
minflow_raw = read_ts(EXTENDED / "Constraint_min_flow.csv")

uc = pd.read_csv(KERNEL / "Unit_commitment_decisions.csv")
test_map = pd.read_csv(ROOT / "prediction_mapping.csv")
sample_sub = pd.read_csv(ROOT / "sample_submission.csv")
topo = yaml.safe_load(open(EXTENDED / "Tokke_Vinje_topology.yaml", encoding="utf-8"))

for name, df in [("price", price_raw), ("inflow", inflow_raw), ("volume", volume_raw), ("water value", wv_raw),
                 ("min volume", minvol_raw), ("min flow", minflow_raw), ("UC", uc), ("test map", test_map)]:
    print(f"{name:12s} {df.shape}  NaNs={int(df.isna().sum().sum())}")

## 2. Clean time series

* **Price** switches from hourly to 15-minute resolution on 2025-09-30 (outside our window, but handled anyway):
  resample to hourly mean and reindex to a complete hourly UTC grid.
* **Inflow** is already hourly from 1958 – we keep the full history for a climatology (normal inflow per day-of-year).
* **Volume / water value** are daily (start of day). Hyljelihyl and Vatjern volumes are constant (assumed 50 % full) → no information.
  Several reservoirs share identical water-value series → de-duplicate.
* **Constraints**: keep only columns that actually vary.

In [ ]:
def to_hourly_grid(df, start, end, how="mean"):
    hourly = df.resample("h").mean() if how == "mean" else df
    grid = pd.date_range(start, end, freq="h", tz="UTC")
    out = hourly.reindex(grid)
    missing = out.isna().any(axis=1)
    inside = missing & (out.index >= hourly.index.min()) & (out.index <= hourly.index.max())
    print(f"  gaps inside data range: {int(inside.sum())}, hours outside data range (edge-filled): {int((missing & ~inside).sum())}")
    out = out.interpolate("time", limit_direction="both")
    return out

GRID_START, GRID_END = "2014-12-01", "2025-01-08 23:00"   # covers lookbacks before the first case and the last case's horizon

print("price:");   price = to_hourly_grid(price_raw, GRID_START, GRID_END)["Day-ahead price (EUR/MWh)"].rename("price")
print("inflow:");  inflow = to_hourly_grid(inflow_raw, GRID_START, GRID_END)
print("minvol:");  minvol = to_hourly_grid(minvol_raw, "2015-01-01", GRID_END)
print("minflow:"); minflow = to_hourly_grid(minflow_raw, "2015-01-01", GRID_END)

price.describe()

In [ ]:
def drop_constant(df, label):
    const = [c for c in df.columns if df[c].nunique() <= 1]
    if const:
        print(f"{label}: dropping constant columns {const}")
    return df.drop(columns=const)

def dedupe_columns(df, label):
    keep, groups = [], {}
    for c in df.columns:
        twin = next((k for k in keep if df[k].equals(df[c])), None)
        if twin is None:
            keep.append(c); groups[c] = [c]
        else:
            groups[twin].append(c)
    print(f"{label}: {df.shape[1]} columns -> {len(keep)} unique series")
    for k, g in groups.items():
        if len(g) > 1:
            print(f"   {k:14s} == {g[1:]}")
    return df[keep], groups

volume = drop_constant(volume_raw, "volume")
wv, wv_groups = dedupe_columns(wv_raw, "water value")
minvol = drop_constant(minvol, "min volume")
minflow = drop_constant(minflow, "min flow")
inflow = drop_constant(inflow, "inflow")

# sanity: daily series must be complete
for name, df in [("volume", volume), ("water value", wv)]:
    gaps = df.index.to_series().diff().dropna().ne(pd.Timedelta("1D")).sum()
    print(f"{name}: {df.index.min().date()} .. {df.index.max().date()}, gaps={gaps}")

## 3. Topology: which reservoir feeds which plant?

From the YAML connections we derive, for each plant, the reservoir it draws from (upstream, possibly via a tunnel)
and the reservoir it discharges into (downstream). The economic intuition from hydropower scheduling:
a unit tends to run when the **price exceeds the water value** of its upstream reservoir
(net of the value the water still has downstream).

In [ ]:
model = topo["model"]
max_vol = {r: v["max_vol"] for r, v in model["reservoir"].items()}
gen_info = pd.DataFrame(model["generator"]).T[["p_min", "p_max", "startcost_const"]].astype(float)

conns = pd.DataFrame(topo["connections"])
def follow_to_reservoir(node, direction):
    """Walk through tunnels/rivers until a reservoir is reached."""
    key, other, other_type = ("to", "from", "from_type") if direction == "up" else ("from", "to", "to_type")
    frontier, seen = [node], set()
    found = []
    while frontier:
        n = frontier.pop()
        for _, c in conns[conns[key] == n].iterrows():
            if c[other_type] == "reservoir":
                found.append(c[other])
            elif c[other_type] in ("tunnel", "river") and c[other] not in seen:
                seen.add(c[other]); frontier.append(c[other])
    return found

plants = list(model["plant"].keys())
plant_map = {p: {"up": follow_to_reservoir(p, "up"), "down": follow_to_reservoir(p, "down")} for p in plants}
for p, m in plant_map.items():
    print(f"{p:12s} up={m['up']}  down={m['down']}")

target_cols = [c for c in sample_sub.columns if c != "Run No"]
UNITS = list(dict.fromkeys(re.match(r"result_committed_(.+)_t\d+$", c).group(1) for c in target_cols))
unit_plant = {u: u.rsplit("_", 1)[0] for u in UNITS}
print(len(UNITS), "units:", UNITS)

In [ ]:
wv_full = wv_raw  # water value per *original* reservoir name (before de-duplication)

def plant_wv(day_index):
    """Water value of upstream (mean over feeding reservoirs) and downstream reservoir per plant."""
    out = {}
    for p, m in plant_map.items():
        out[f"wv_up_{p}"] = wv_full.loc[day_index, m["up"]].mean(axis=1).to_numpy()
        out[f"wv_down_{p}"] = (wv_full.loc[day_index, m["down"]].mean(axis=1).to_numpy()
                               if m["down"] else np.zeros(len(day_index)))   # Hogga discharges to sea
    return out

## 4. Cases (train + test)

In [ ]:
train_cases = uc[["Run No", "starttime"]].assign(split="train")
test_cases = test_map[["Run No", "starttime"]].assign(split="test")
cases = pd.concat([train_cases, test_cases], ignore_index=True)
cases["starttime"] = pd.to_datetime(cases["starttime"], utc=True)
cases = cases.sort_values("Run No").reset_index(drop=True)

assert cases["Run No"].is_unique and cases["starttime"].diff().dropna().eq(pd.Timedelta("1D")).all()
assert set(test_map["Run No"]) == set(sample_sub["Run No"])
print(cases.groupby("split")["starttime"].agg(["min", "max", "count"]))

# labels -> array (N_train, 14, 168) in the exact submission column order
Y_train = uc.set_index("Run No")[target_cols].to_numpy(dtype=np.float32).reshape(len(uc), len(UNITS), H)
print("Y_train", Y_train.shape, "positive rate", Y_train.mean().round(3))

### Hourly windows
For every case we cut the 168-hour window `[starttime, starttime + 168h)` out of each hourly series
with vectorised indexing.

In [ ]:
def windows(series_or_df, starts, lookback=0):
    """Return array (n_cases, H + lookback, n_cols) of hourly values for each case window."""
    df = series_or_df.to_frame() if isinstance(series_or_df, pd.Series) else series_or_df
    pos = df.index.get_indexer(starts)
    assert (pos >= lookback).all(), "window starts outside the hourly grid"
    idx = pos[:, None] + np.arange(-lookback, H)[None, :]
    assert idx.max() < len(df), "window ends outside the hourly grid"
    return df.to_numpy(dtype=np.float32)[idx]

starts = pd.DatetimeIndex(cases["starttime"])
N = len(cases)
P = windows(price, starts)[..., 0]                 # (N, 168)
P_prev = windows(price, starts, lookback=H)[:, :H, 0]  # the week before the case
INF = windows(inflow, starts)                       # (N, 168, n_inflow)
MINV = windows(minvol, starts) if minvol.shape[1] else np.zeros((N, H, 0), np.float32)
MINF = windows(minflow, starts) if minflow.shape[1] else np.zeros((N, H, 0), np.float32)
print(P.shape, INF.shape, MINV.shape, MINF.shape)

## 5. Case-level (static) features

* calendar (month, day-of-year as sin/cos, year)
* price statistics for the week + previous week
* initial reservoir volume (absolute and fraction of `max_vol`), optional end-of-week volume change
* water value at start and at end of horizon (`start + 7 days`, as described in the dataset doc)
* per plant: upstream/downstream water value, price – water value spread, share of hours where price > water value
* inflow: weekly mean per reservoir, total, and anomaly vs. 1958–2014 climatology
* constraints: weekly mean of minimum flow / volume

In [ ]:
day0 = starts.normalize()
day_end = day0 + pd.Timedelta("7D")
assert day_end.max() <= volume.index.max(), "end-of-horizon water value / volume not available"

f = {"Run No": cases["Run No"].to_numpy()}
f["year"] = starts.year
f["month"] = starts.month
doy = starts.dayofyear.to_numpy()
f["doy_sin"], f["doy_cos"] = np.sin(2 * np.pi * doy / 365.25), np.cos(2 * np.pi * doy / 365.25)

# price
f["price_mean"], f["price_std"] = P.mean(1), P.std(1)
f["price_min"], f["price_max"] = P.min(1), P.max(1)
f["price_q10"], f["price_q90"] = np.quantile(P, 0.1, axis=1), np.quantile(P, 0.9, axis=1)
f["price_prev_week_mean"] = P_prev.mean(1)
f["price_week_over_week"] = f["price_mean"] - f["price_prev_week_mean"]
f["price_daily_amplitude"] = (P.reshape(N, 7, 24).max(2) - P.reshape(N, 7, 24).min(2)).mean(1)

# volumes
for r in volume.columns:
    v0 = volume.loc[day0, r].to_numpy()
    f[f"vol_{r}"] = v0
    f[f"fill_{r}"] = v0 / max_vol[r]
    if USE_END_OF_WEEK_VOLUME:
        f[f"dvol_{r}"] = volume.loc[day_end, r].to_numpy() - v0
f["fill_total"] = sum(volume.loc[day0, r].to_numpy() for r in volume.columns) / sum(max_vol[r] for r in volume.columns)

# water values (unique series only)
for r in wv.columns:
    f[f"wv0_{r}"] = wv.loc[day0, r].to_numpy()
    f[f"wv7_{r}"] = wv.loc[day_end, r].to_numpy()
    f[f"dwv_{r}"] = f[f"wv7_{r}"] - f[f"wv0_{r}"]

# plant-level economics (end-of-horizon water value drives the weekly optimisation)
pw_end = plant_wv(day_end)
for p in plants:
    up, down = pw_end[f"wv_up_{p}"], pw_end[f"wv_down_{p}"]
    f[f"wvup_{p}"], f[f"wvnet_{p}"] = up, up - down
    f[f"spread_{p}"] = f["price_mean"] - up
    f[f"frac_price_gt_wv_{p}"] = (P > up[:, None]).mean(1)

# inflow + climatology anomaly (climatology from years before the training period)
clim_src = inflow_raw.loc[:"2014-12-31", inflow.columns]
clim = clim_src.groupby(clim_src.index.dayofyear).mean()
inf_week = INF.mean(1)                                     # (N, n_inflow)
# climatology of the 7 days in the window
clim_days = ((doy[:, None] - 1 + np.arange(7)[None, :]) % 365) + 1          # (N, 7)
clim_week = clim.to_numpy()[clim_days - 1].mean(1)                           # (N, n_inflow)
for j, r in enumerate(inflow.columns):
    f[f"inflow_{r}"] = inf_week[:, j]
f["inflow_total"] = inf_week.sum(1)
f["inflow_total_anomaly"] = inf_week.sum(1) / (clim_week.sum(1) + 1e-6)

for j, r in enumerate(minflow.columns):
    f[f"minflow_{r}"] = MINF[:, :, j].mean(1)
for j, r in enumerate(minvol.columns):
    f[f"minvol_{r}"] = MINV[:, :, j].mean(1)

case_feat = pd.DataFrame(f)
case_feat.insert(1, "starttime", cases["starttime"].to_numpy())
case_feat.insert(2, "split", cases["split"].to_numpy())
# fold for time-based CV on train (one fold per year). Neighbouring cases overlap by 6 days,
# so random K-fold would leak – validate on whole years (optionally with a 7-day gap).
case_feat["fold"] = np.where(case_feat["split"] == "train", case_feat["year"], -1)
print(case_feat.shape)
case_feat.head()

## 6. Hourly features

One row per (case, hour `t`). Includes position in the horizon (end-of-horizon effects matter in optimisation),
local hour/weekday, price shape features inside the week, and price minus upstream water value per plant.

In [ ]:
t = np.arange(H)
ts = starts.tz_convert(None).to_numpy()[:, None] + (t * np.timedelta64(1, "h"))[None, :]      # (N, 168) UTC timestamps
ts_local = pd.DatetimeIndex(ts.ravel()).tz_localize("UTC").tz_convert(TZ_LOCAL)

hf = {
    "Run No": np.repeat(cases["Run No"].to_numpy(), H),
    "t": np.tile(t, N),
    "day_in_horizon": np.tile(t // 24, N),
    "hour_local": ts_local.hour.to_numpy(),
    "dow_local": ts_local.dayofweek.to_numpy(),
    "is_weekend": (ts_local.dayofweek >= 5).astype(np.int8),
}
hs = {}   # hourly arrays (N,168) – reused for the sequence tensor
hs["price"] = P
hs["price_minus_mean"] = P - P.mean(1, keepdims=True)
hs["price_z"] = hs["price_minus_mean"] / (P.std(1, keepdims=True) + 1e-6)
hs["price_rank_week"] = P.argsort(1).argsort(1) / (H - 1)
Pd = P.reshape(N, 7, 24)
hs["price_rank_day"] = (Pd.argsort(2).argsort(2) / 23).reshape(N, H)
hs["price_minus_daymean"] = (Pd - Pd.mean(2, keepdims=True)).reshape(N, H)
hs["price_diff_prev"] = np.diff(P, axis=1, prepend=P_prev[:, -1:])
hs["price_diff_next"] = np.concatenate([P[:, 1:] - P[:, :-1], np.zeros((N, 1), np.float32)], axis=1)
k = 6
pad = np.pad(P, ((0, 0), (k, k)), mode="edge")
hs["price_roll13"] = np.stack([pad[:, i:i + H] for i in range(2 * k + 1)]).mean(0)
for p in plants:
    hs[f"price_minus_wvup_{p}"] = P - pw_end[f"wv_up_{p}"][:, None]
hs["inflow_total_h"] = INF.sum(2)
for j, r in enumerate(minflow.columns):
    hs[f"minflow_h_{r}"] = MINF[:, :, j]
for j, r in enumerate(minvol.columns):
    hs[f"minvol_h_{r}"] = MINV[:, :, j]

for name, arr in hs.items():
    hf[name] = arr.astype(np.float32).ravel()
hourly = pd.DataFrame(hf)

# targets: y_<unit>, NaN for test cases
Y_all = np.full((N, len(UNITS), H), np.nan, dtype=np.float32)
train_mask = (cases["split"] == "train").to_numpy()
assert np.array_equal(cases.loc[train_mask, "Run No"].to_numpy(), uc["Run No"].to_numpy())
Y_all[train_mask] = Y_train
for u_i, u in enumerate(UNITS):
    hourly[f"y_{u}"] = Y_all[:, u_i, :].ravel()

hourly = hourly.merge(case_feat, on="Run No", how="left")
print(hourly.shape, f"{hourly.memory_usage(deep=True).sum() / 1e6:.0f} MB")
hourly.head()

## 7. Sanity checks

In [ ]:
# 1) target reshaping round-trips to the original label columns
chk = hourly[hourly["split"] == "train"].pivot(index="Run No", columns="t", values=f"y_{UNITS[3]}")
orig = uc.set_index("Run No")[[f"result_committed_{UNITS[3]}_t{i}" for i in range(H)]]
assert np.array_equal(chk.to_numpy(), orig.to_numpy()), "label reshaping mismatch"

# 2) price window matches a direct lookup
r = cases.sample(1, random_state=0).iloc[0]
direct = price.loc[r["starttime"]: r["starttime"] + pd.Timedelta(hours=H - 1)].to_numpy()
assert np.allclose(direct, P[cases.index[cases["Run No"] == r["Run No"]][0]])

# 3) no NaN in features
feat_nan = hourly.drop(columns=[c for c in hourly if c.startswith("y_")]).isna().sum()
assert feat_nan.sum() == 0, feat_nan[feat_nan > 0]

# 4) intuition check: units are ON more often when price exceeds the upstream water value
tr = hourly[hourly["split"] == "train"]
rows = []
for u in UNITS:
    above = tr[f"price_minus_wvup_{unit_plant[u]}"] > 0
    rows.append((u, tr.loc[above, f"y_{u}"].mean(), tr.loc[~above, f"y_{u}"].mean()))
pd.DataFrame(rows, columns=["unit", "P(on | price>wv)", "P(on | price<=wv)"]).round(3)

## 8. Save

In [ ]:
hourly_feature_cols = list(hs.keys()) + ["t", "day_in_horizon", "hour_local", "dow_local", "is_weekend"]
static_feature_cols = [c for c in case_feat.columns if c not in ("Run No", "starttime", "split", "fold")]
y_cols = [f"y_{u}" for u in UNITS]

FLOAT_FMT = "%.6g"   # 6 significant digits keeps the files small without losing useful precision

case_feat.to_csv(OUT / "cases.csv", index=False, float_format=FLOAT_FMT)

hourly_cols = ["Run No"] + hourly_feature_cols
is_train_row = hourly["split"] == "train"
hourly.loc[is_train_row, hourly_cols + y_cols].astype({c: "int8" for c in y_cols})       .to_csv(OUT / "hourly_train.csv", index=False, float_format=FLOAT_FMT)
hourly.loc[~is_train_row, hourly_cols].to_csv(OUT / "hourly_test.csv", index=False, float_format=FLOAT_FMT)

meta = {
    "units": UNITS,
    "unit_plant": unit_plant,
    "plant_map": plant_map,
    "submission_columns": sample_sub.columns.tolist(),
    "hourly_feature_cols": hourly_feature_cols,
    "static_feature_cols": static_feature_cols,
    "y_cols": y_cols,
    "use_end_of_week_volume": USE_END_OF_WEEK_VOLUME,
}
(OUT / "meta.json").write_text(json.dumps(meta, indent=2))
for p in sorted(OUT.iterdir()):
    print(f"{p.name:18s} {p.stat().st_size / 1e6:7.1f} MB")

## 9. From predictions back to a Kaggle submission

Models trained on the hourly CSV files predict one probability per (case, hour, unit). This helper puts them back into
the `sample_submission.csv` layout (`result_committed_<unit>_t<hour>`), which is required by `kaggle_metric.py`.

In [ ]:
def to_submission(pred, run_nos):
    """pred: array (n_cases, 14, 168) in UNITS order, run_nos: matching Run No values."""
    cols = [f"result_committed_{u}_t{i}" for u in UNITS for i in range(H)]
    sub = pd.DataFrame(pred.reshape(len(run_nos), -1), columns=cols)
    sub.insert(0, "Run No", run_nos)
    return sub.set_index("Run No").loc[sample_sub["Run No"]].reset_index()[sample_sub.columns]

# baseline check: per-unit, per-hour-of-day ON frequency from training data
base = tr.groupby("hour_local")[y_cols].mean()
te = hourly[hourly["split"] == "test"]
pred = np.stack([te["hour_local"].map(base[c]).to_numpy().reshape(-1, H) for c in y_cols], axis=1)
sub = to_submission(pred, te["Run No"].to_numpy()[::H])
assert sub.columns.equals(sample_sub.columns) and len(sub) == len(sample_sub)
sub.iloc[:3, :6]